# TransJakarta Demand Predictor — Preprocessing

Notebook ini mengubah **transaksi tap-in mentah** menjadi dataset jumlah transaksi per koridor per jam.

## Tujuan kita

Kita ingin menjawab pertanyaan:

> "Untuk koridor dan arah tertentu, berapa banyak transaksi tap-in yang terjadi pada jam berikutnya?"

**Penting:** `passenger_count` di sini berarti jumlah transaksi tap-in yang tercatat, bukan jumlah penumpang yang sedang berada di dalam bus. Dataset ini juga bukan data real-time.

## Cara menjalankan

1. Jalankan notebook dari folder `ngawibarat` atau folder lain yang dapat menemukan folder `data/raw` melalui jalur proyek.
2. Jalankan setiap cell dari atas ke bawah.
3. Jika library belum tersedia, jalankan cell instalasi.
4. Periksa hasil audit sebelum memakai dataset untuk eksperimen model.

## 0. Persiapan

Kita memakai Python, pandas untuk mengolah tabel, dan matplotlib untuk grafik.

In [ ]:
# Jika library belum terpasang, hapus tanda komentar pada baris berikut:
# %pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (parent for parent in [start, *start.parents]
     if (parent / "data" / "raw" / "transjakarta.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root with data/raw/transjakarta.csv not found")
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "transjakarta.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(DATA_PATH)
print("Ukuran data mentah:", df.shape)
display(df.head())

## 1. Kenali data mentah

Satu baris pada file ini mewakili satu transaksi. Kita periksa nama kolom, tipe data, duplikasi, dan nilai kosong.

Kolom yang paling penting untuk proyek kita:
- `transID`: identitas transaksi
- `corridorID`: identitas koridor
- `direction`: arah perjalanan
- `tapInTime`: waktu transaksi tap-in

Kita tidak membutuhkan informasi kartu atau identitas pribadi untuk menghitung jumlah transaksi per jam.

In [ ]:
print("Nama kolom:")
print(df.columns.tolist())

print("\\nTipe data:")
display(df.dtypes.to_frame("dtype"))

print("\\nJumlah missing value per kolom:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))

print("Duplikasi seluruh baris:", df.duplicated().sum())
if "transID" in df.columns:
    print("Duplikasi transID:", df["transID"].duplicated().sum())

## 2. Pilih kolom yang diperlukan dan bersihkan timestamp

Kita hanya mempertahankan kolom yang diperlukan. Baris tanpa `corridorID` atau `tapInTime` tidak bisa dimasukkan ke hitungan per koridor per jam, sehingga dikeluarkan dari agregasi.

Missing `tapOutTime` tidak menjadi alasan untuk membuang transaksi, karena target kita dihitung dari waktu tap-in.

In [ ]:
required = ["corridorID", "direction", "tapInTime"]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

work = df[required].copy()
work["tapInTime"] = pd.to_datetime(work["tapInTime"], errors="coerce")

before = len(work)
work = work.dropna(subset=["corridorID", "tapInTime"]).copy()
work["corridorID"] = work["corridorID"].astype(str).str.strip()
work = work[work["corridorID"] != ""].copy()

print("Baris awal:", before)
print("Baris yang bisa dipakai:", len(work))
print("Baris dikeluarkan karena corridorID/tapInTime kosong atau invalid:", before - len(work))
print("Rentang waktu:", work["tapInTime"].min(), "sampai", work["tapInTime"].max())
print("Jumlah corridorID unik:", work["corridorID"].nunique())
print("Jumlah direction unik (termasuk missing):", work["direction"].nunique(dropna=True))

## 3. Agregasi transaksi menjadi jumlah per jam

Kita kelompokkan transaksi berdasarkan:
- tanggal
- koridor
- arah
- jam

Lalu hitung jumlah transaksi tap-in di setiap kelompok. Inilah target awal kita, `passenger_count`.

Catatan: kelompok yang muncul di hasil ini memiliki setidaknya satu transaksi. Kelompok yang tidak muncul belum otomatis dianggap nol.

In [ ]:
work["date"] = work["tapInTime"].dt.floor("D")
work["hour"] = work["tapInTime"].dt.hour

hourly_observed = (
    work.groupby(["date", "corridorID", "direction", "hour"], dropna=False)
        .size()
        .reset_index(name="passenger_count")
        .sort_values(["corridorID", "direction", "date", "hour"])
        .reset_index(drop=True)
)

hourly_observed["day_of_week"] = hourly_observed["date"].dt.dayofweek
hourly_observed["is_weekend"] = (hourly_observed["day_of_week"] >= 5).astype(int)

print("Jumlah baris agregasi dengan transaksi:", len(hourly_observed))
display(hourly_observed.head(10))

## 4. Periksa pola waktu

Grafik berikut menjumlahkan transaksi semua koridor berdasarkan jam. Ini membantu melihat pola umum, tetapi bukan perbandingan kapasitas atau tingkat kepadatan tiap halte.

In [ ]:
by_hour = hourly_observed.groupby("hour")["passenger_count"].sum().reindex(range(24), fill_value=0)

plt.figure(figsize=(11, 4))
by_hour.plot(kind="bar")
plt.title("Jumlah transaksi tap-in menurut jam (seluruh koridor)")
plt.xlabel("Jam")
plt.ylabel("Jumlah transaksi")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(by_hour.rename("transactions").to_frame())

## 5. Buat panel waktu per koridor dan arah — tanpa mengarang nilai nol

Untuk menghitung fitur seperti jumlah transaksi satu jam sebelumnya (`lag_1`), kita perlu tahu apakah jam sebelumnya tercatat.

Kita buat rangkaian waktu per koridor dan arah. Jika kombinasi waktu tidak ditemukan pada file, `passenger_count` dibiarkan **NaN (belum diketahui)**, bukan langsung diisi nol. Ini lebih hati-hati karena kita belum membuktikan apakah tidak ada transaksi atau datanya tidak terekam.

Untuk model, baris dengan target yang benar-benar teramati dan fitur lag yang tersedia akan digunakan.

In [ ]:
# Buat kombinasi waktu per koridor-arah pada rentang data yang tersedia.
# Missing hour tetap NaN; tidak diubah menjadi 0.
panel_parts = []

for (corridor, direction), group in hourly_observed.groupby(["corridorID", "direction"], dropna=False):
    group = group.sort_values(["date", "hour"]).copy()
    start = group["date"].min()
    end = group["date"].max()

    full_hours = pd.date_range(
        start=start,
        end=end + pd.Timedelta(hours=23),
        freq="h"
    )

    temp = group.set_index(group["date"] + pd.to_timedelta(group["hour"], unit="h"))
    temp = temp[["passenger_count"]].reindex(full_hours)
    temp.index.name = "timestamp"
    temp = temp.reset_index()
    temp["corridorID"] = corridor
    temp["direction"] = direction
    panel_parts.append(temp)

panel = pd.concat(panel_parts, ignore_index=True)
panel["date"] = panel["timestamp"].dt.floor("D")
panel["hour"] = panel["timestamp"].dt.hour
panel["day_of_week"] = panel["timestamp"].dt.dayofweek
panel["is_weekend"] = (panel["day_of_week"] >= 5).astype(int)

print("Ukuran panel lengkap:", panel.shape)
print("Jam dengan jumlah transaksi teramati:", panel["passenger_count"].notna().sum())
print("Jam yang tidak tercatat (NaN, belum diasumsikan nol):", panel["passenger_count"].isna().sum())
display(panel.head(10))

## 6. Feature engineering: lag dan rolling

- `lag_1`: jumlah transaksi pada satu jam sebelumnya.
- `lag_24`: jumlah transaksi pada 24 jam sebelumnya.
- `rolling_3`: rata-rata transaksi tiga jam sebelumnya, tidak termasuk jam saat ini.

Fitur dihitung terpisah untuk setiap koridor dan arah. Karena missing hour masih NaN, fitur yang membutuhkan jam tersebut juga akan menjadi NaN.

In [ ]:
panel = panel.sort_values(["corridorID", "direction", "timestamp"]).copy()
group_keys = ["corridorID", "direction"]

panel["lag_1"] = panel.groupby(group_keys)["passenger_count"].shift(1)
panel["lag_24"] = panel.groupby(group_keys)["passenger_count"].shift(24)
panel["rolling_3"] = (
    panel.groupby(group_keys)["passenger_count"]
         .transform(lambda s: s.shift(1).rolling(window=3, min_periods=3).mean())
)

feature_cols = [
    "corridorID", "direction", "hour", "day_of_week", "is_weekend",
    "lag_1", "lag_24", "rolling_3"
]
target_col = "passenger_count"

model_data = panel.dropna(subset=[target_col, "lag_1", "lag_24", "rolling_3"]).copy()
print("Baris dengan target dan seluruh fitur lag tersedia:", len(model_data))
display(model_data[feature_cols + [target_col]].head())

## 7. Simpan hasil preprocessing

Kita simpan tiga file:
1. `transjakarta_hourly_observed.csv`: hanya kelompok tanggal-koridor-arah-jam yang memiliki transaksi.
2. `transjakarta_hourly_panel.csv`: panel jam lengkap, nilai yang tidak tercatat tetap kosong.
3. `transjakarta_model_ready.csv`: baris dengan target dan fitur lag lengkap, siap untuk eksperimen model awal.

Jangan menghapus file panel karena penting untuk audit asumsi missing hour.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
hourly_observed.to_csv(PROCESSED_DIR / "transjakarta_hourly_observed.csv", index=False)
panel.to_csv(PROCESSED_DIR / "transjakarta_hourly_panel.csv", index=False)
model_data[feature_cols + [target_col, "date", "timestamp"]].to_csv(
    PROCESSED_DIR / "transjakarta_model_ready.csv", index=False
)

print("File berhasil disimpan:")
print("- transjakarta_hourly_observed.csv")
print("- transjakarta_hourly_panel.csv")
print("- transjakarta_model_ready.csv")

## 8. Apa yang sudah dan belum kita lakukan?

### Sudah
- Membaca transaksi mentah.
- Memeriksa missing value dan duplikasi.
- Mengambil transaksi dengan koridor dan timestamp yang valid.
- Menghitung transaksi tap-in per tanggal, koridor, arah, dan jam.
- Membuat fitur kalender dan lag.
- Menyimpan hasil preprocessing.

### Belum
- Memastikan apakah jam tanpa transaksi memang berarti nol penumpang atau data tidak terekam.
- Melakukan train-test split dan membandingkan model.
- Memvalidasi hasil terhadap periode data yang lebih panjang.
- Mengukur kepadatan aktual bus atau halte.

**Tahap berikutnya:** audit panel dan model-ready dataset, lalu eksperimen baseline dan tiga model regresi dengan pemisahan data berdasarkan waktu.